# US Airline Sentiment & Complaint Analysis

A major **US airline group** receives thousands of customer tweets daily. The customer experience team needs to automatically classify each tweet as **positive** or **negative** and identify **recurring complaint themes** within negative tweets.

Standard text preprocessing strips out common words like `not`, `no`, and `never` — but these words carry sentiment. Tweets like *"The service was not bad"* get misclassified as **Negative** when negation words are removed. Your pipeline must clean the text **without losing words that affect meaning**.

---

**Your Task:**

Build a sentiment classification and theme discovery pipeline using the provided `train.csv` and `test.csv`.

**Part A — Sentiment Classification**

1. **Preprocess** — remove mentions, URLs, hashtags, digits, punctuation. Lowercase, remove stopwords (preserve negations), lemmatize.
2. **Vectorize** — convert cleaned text to features using `TfidfVectorizer` or `CountVectorizer`. Fit on training data only.
3. **Train** — fit a `MultinomialNB` classifier.
4. **Evaluate** — report accuracy and macro precision on the test set.

**Part B — Theme Discovery**

5. **Filter** — select negative tweets from training data.
6. **Embed** — train `Word2Vec` on cleaned training tweets (`vector_size=100`).
7. **Represent** — average word vectors to create a sentence vector per negative tweet.
8. **Cluster** — apply `KMeans` (`n_clusters` from `{3, 4, 5}`) on the sentence vectors.

---

### Dataset

| Column | Description |
|--------|-------------|
| `text` | Raw tweet |
| `airline` | Airline name |
| `label` | `0` = negative, `1` = positive |

---

### Important Instructions

- Wrap your preprocessing in a function named `preprocess_pipeline(text)` — this exact name is required.
- Negation words (`not`, `no`, `never`) must survive preprocessing.

In [2]:
# Run this cell before writing your solution

import os
os.system("pip install numpy --upgrade -q")

import re
import warnings
warnings.filterwarnings("ignore")

import numpy as np
import pandas as pd

import nltk
from nltk.corpus import stopwords
from nltk.stem import WordNetLemmatizer

from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.naive_bayes import MultinomialNB
from sklearn.cluster import KMeans
from sklearn.metrics import accuracy_score, precision_score

from gensim.models import Word2Vec

nltk.download("stopwords", quiet=True)
nltk.download("wordnet", quiet=True)
nltk.download("omw-1.4", quiet=True)

train_df = pd.read_csv("https://s3.ap-south-1.amazonaws.com/new-assets.ccbp.in/frontend/content/aiml/classical-ml/train.csv")
test_df = pd.read_csv("https://s3.ap-south-1.amazonaws.com/new-assets.ccbp.in/frontend/content/aiml/classical-ml/test.csv")

print(f"Train: {train_df.shape}")
print(f"Test: {test_df.shape}")
train_df.head()


[notice] A new release of pip is available: 26.1.2 -> 26.2.1
[notice] To update, run: pip install --upgrade pip


Train: (9232, 3)
Test: (2309, 3)


,text,airline,label
0,@JetBlue Finally starting to come out. 40 minu...,Delta,0
1,@USAirways oh I got right through to an agent....,US Airways,0
2,@AmericanAir any idea on what the wait time is...,American,0
3,@AmericanAir Priority baggage evidently means ...,American,0
4,@USAirways you're really fudgin up with all th...,US Airways,0


Part A: Sentiment Classification

1. Preprocessing Pipeline definition

In [3]:
import string
lemmatizer = WordNetLemmatizer()

#negation terms
stop_words = set(stopwords.words('english'))
negations = {'not', 'no', 'never'}
custom_stopwords = stop_words - negations

def preprocess_pipeline(text):
  if not isinstance(text, str):
    return ''

  #Remove URLs
  text = re.sub(r'http\S+|www\S+|https\S+', '', text, flags=re.MULTILINE)
  #Remove mentions
  text = re.sub(r'(@\w+|#\w+)', '', text)
  #Remove digits
  text = re.sub(r'\d+', '', text)
  #Remove punctuation
  text = text.translate(str.maketrans('', '', string.punctuation))
  #Lowercase
  text = text.lower()

  #Tokenize, remove non-negation stopwords and lemmatize
  tokens = text.split()
  clean_tokens = [
      lemmatizer.lemmatize(token)
      for token in tokens
      if token not in custom_stopwords and len(token) > 0
  ]

  return ' '.join(clean_tokens)

#Apply preprocessing to traning and test sets
train_df['cleaned_text'] = train_df['text'].apply(preprocess_pipeline)
test_df['cleaned_text'] = test_df['text'].apply(preprocess_pipeline)

2. Vectorize using TfidfVectorizer

In [9]:
vectorizer = TfidfVectorizer()
X_train = vectorizer.fit_transform(train_df['cleaned_text'])
X_test = vectorizer.transform(test_df['cleaned_text'])

y_train = train_df['label']
y_test = test_df['label']
X_train
X_test

<2309x7915 sparse matrix of type '<class 'numpy.float64'>'
	with 19897 stored elements in Compressed Sparse Row format>

3. Train a MultinomialNB classifier

In [10]:
nb_classifer = MultinomialNB()
nb_classifer.fit(X_train, y_train)
nb_classifer

,"alpha alpha: float or array-like of shape (n_features,), default=1.0Additive (Laplace/Lidstone) smoothing parameter(set alpha=0 and force_alpha=True, for no smoothing).",1.0
,"force_alpha force_alpha: bool, default=TrueIf False and alpha is less than 1e-10, it will set alpha to1e-10. If True, alpha will remain unchanged. This may causenumerical errors if alpha is too close to 0... versionadded:: 1.2.. versionchanged:: 1.4 The default value of `force_alpha` changed to `True`.",True
,"fit_prior fit_prior: bool, default=TrueWhether to learn class prior probabilities or not.If false, a uniform prior will be used.",True
,"class_prior class_prior: array-like of shape (n_classes,), default=NonePrior probabilities of the classes. If specified, the priors are notadjusted according to the data.",None
Name,Type,Value
"class_count_ class_count_: ndarray of shape (n_classes,)Number of samples encountered for each class during fitting. Thisvalue is weighted by the sample weight when provided.","ndarray[float64](2,)","[7342.,1890.]"
"class_log_prior_ class_log_prior_: ndarray of shape (n_classes,)Smoothed empirical log probability for each class.","ndarray[float64](2,)","[-0.23,-1.59]"
"classes_ classes_: ndarray of shape (n_classes,)Class labels known to the classifier","ndarray[int64](2,)","[0,1]"
"feature_count_ feature_count_: ndarray of shape (n_classes, n_features)Number of samples encountered for each (class, feature)during fitting. This value is weighted by the sample weight whenprovided.","ndarray[float64](2, 7915)","[[35.07, 0.36, 0. ,..., 0.37, 0.34, 1.07], [ 5.49, 0. , 0.37,..., 0. , 0. , 0. ]]"
"feature_log_prob_ feature_log_prob_: ndarray of shape (n_classes, n_features)Empirical log probability of featuresgiven a class, ``P(x_i|y)``.","ndarray[float64](2, 7915)","[[ -6.7 , -9.97,-10.28,..., -9.97, -9.99, -9.56], [ -7.56, -9.43, -9.11,..., -9.43, -9.43, -9.43]]"
n_features_in_ n_features_in_: intNumber of features seen during :term:`fit`... versionadded:: 0.24,int,7915


4. Evaluate: Accuracy and Macro Precision

In [11]:
y_preds = nb_classifer.predict(X_test)
acc = accuracy_score(y_test, y_preds)
macro_precision = precision_score(y_test, y_preds, average='macro')

print(f'Test Accuracy: {acc:.4f}')
print(f'Test Macro Precision: {macro_precision:.4f}')

Test Accuracy: 0.8476
Test Macro Precision: 0.9157


Part B: Theme Discovery

1. Filter: Select negative tweets from training data

In [15]:
negative_tweets = train_df[train_df['label'] == 0]['cleaned_text']
negative_tokenized = [
  tweet.split() for tweet in negative_tweets if len(tweet.split()) > 0
]

negative_tweets

0       finally starting come minute no communication ...
1       oh got right agent kept putting hold still did...
2          idea wait time refund told day phone well past
3               priority baggage evidently mean come last
4                             youre really fudgin delay 😡
                              ...                        
9227    money change flight dont answer phone suggesti...
9228    u costing get home today no reason send receip...
9229              no flight philly system wide tech issue
9230    customer service suck hang waiting hour talkin...
9231                  paid seat expect able use full seat
Name: cleaned_text, Length: 7342, dtype: str

2. Embed: Train Word2Vec on cleaned training tweets

In [16]:
w2v_model = Word2Vec(
  sentences=negative_tokenized,
  vector_size=100,
  window=5,
  min_count=1,
  workers=4,
  seed=42,
)

w2v_model

3. Represent: Average word vectors to create a sentence vector per negative tweet

In [18]:
def get_sentence_vector(tokens, model, vector_size = 100):
  vectors = [model.wv[word] for word in tokens if word in model.wv]
  if len(vectors) > 0:
    return np.mean(vectors, axis=0)
  return np.zeros(vector_size)

sentence_vectors = np.array(
  [
    get_sentence_vector(tokens, w2v_model, vector_size=100)
    for tokens in negative_tokenized
  ]
)

sentence_vectors

array([[ 0.18708858,  0.0639842 , -0.27584752, ...,  0.31130356,
        -0.346387  , -0.2078677 ],
       [ 0.21927443,  0.08052392, -0.32476488, ...,  0.3750124 ,
        -0.39782125, -0.23697585],
       [ 0.24049139,  0.07798395, -0.34450412, ...,  0.39453292,
        -0.4318781 , -0.25894883],
       ...,
       [ 0.19047102,  0.07439626, -0.29987508, ...,  0.35329294,
        -0.36841494, -0.22136071],
       [ 0.21183035,  0.08078629, -0.33093897, ...,  0.34070617,
        -0.43020794, -0.24037662],
       [ 0.23710997,  0.07735063, -0.34287927, ...,  0.38813654,
        -0.42111704, -0.2426981 ]], dtype=float32)

4. Cluster: Apply KMeans

In [19]:
Kmeans = KMeans(n_clusters=4, random_state=42, n_init=10)
Kmeans.fit(sentence_vectors)
print(f'KMeans clustering complete. Found {len(set(Kmeans.labels_))} clusters.')

KMeans clustering complete. Found 4 clusters.
